# Remote UCB optimization

This notebook runs an `UpperConfidenceBoundGenerator` on a remote server using `HTTPGenerator`. The Xopt object, evaluator, and data stay in this notebook. Only the generator runs on the server. See the [remote generator guide](http_generator_guide.md) for setup details.

**Before running:** start the server in a separate terminal, using the same environment as Jupyter. It will not start without a token:

```bash
export XOPT_HTTP_GENERATOR_TOKEN="<a long random string>"
python -m uvicorn xopt.generators.remote.server:app --host 127.0.0.1 --port 8002
```

Set the same `XOPT_HTTP_GENERATOR_TOKEN` in the environment you launch Jupyter from; the notebook reads it from there. The notebook connects to `http://127.0.0.1:8002` by default. To use a server on another machine, set `XOPT_HTTP_URL` (e.g. `http://server-host:8002`) before launching Jupyter, or forward the port with an SSH tunnel.

In [ ]:
import os
import warnings

from xopt import Xopt, VOCS, Evaluator
from xopt.generators.remote.http import HTTPGenerator
from xopt.resources.test_functions.sinusoid_1d import evaluate_sinusoid

warnings.filterwarnings("ignore")
SMOKE_TEST = os.environ.get("SMOKE_TEST", "").lower() in ["1", "true", "yes"]
N_STEPS = 3 if SMOKE_TEST else 8

Configure the remote address. The default `127.0.0.1:8002` is for two independent processes on the same machine; `XOPT_HTTP_URL` overrides it for another node. The notebook **does not start the server**.


In [ ]:
# Start an independent Uvicorn server in another terminal before running.
# This notebook deliberately does not share its Python process with the server.
base_url = os.environ.get("XOPT_HTTP_URL", "http://127.0.0.1:8002")
api_key = os.environ["XOPT_HTTP_GENERATOR_TOKEN"]
print(f"Independent HTTPGenerator server: {base_url}")

## Build the remote generator and Xopt object

`HTTPGenerator` wraps an already-instantiated generator (here, `UpperConfidenceBoundGenerator`) plus the location of the server (`base_url`). `api_key` is the server's token. The generator's configuration (including `vocs`) is sent to the server when the session is first created, and the server builds an instance of the same class from it.

In [ ]:
from xopt.generators.bayesian.upper_confidence_bound import (
    UpperConfidenceBoundGenerator,
)

vocs = VOCS(
    variables={"x1": [0, 6.28]},
    objectives={"y1": "MINIMIZE"},
)

generator = HTTPGenerator(
    base_url=base_url,
    api_key=api_key,
    generator=UpperConfidenceBoundGenerator(vocs=vocs, beta=0.1),
)

evaluator = Evaluator(function=evaluate_sinusoid)
X = Xopt(generator=generator, evaluator=evaluator)
X

## Run optimization

Each `X.step()` call transparently sends the current data to the remote session and asks it to suggest the next point — the optimization loop is identical to running the generator in-process.

In [ ]:
X.random_evaluate(3)
for i in range(N_STEPS):
    print(i)
    X.step()

## View output data

Inspect the collected data. Each row is an evaluated `x1` and its observed `y1`. The data is stored in the notebook's Xopt object.

In [ ]:
X.data

## Visualize the remote model

`visualize_model` isn't part of the remote proxy's interface (only `suggest`/`ingest` are). Accessing it triggers `HTTPGenerator` to pull a full, fitted snapshot of the remote generator (including the GP model) and reconstruct a real local copy, then delegate the call to it. This mirror is a read-only, point-in-time snapshot — call `X.generator.pull()` again after more iterations to refresh it.

In [ ]:
fig, ax = X.generator.visualize_model(n_grid=100)

In [ ]:
X.generator.data

Finalize the **remote session** and free its session state. The separately started Uvicorn service keeps running for the next notebook.


In [ ]:
# Finalize the remote session. The independently launched server stays running.
X.generator.finalize()
print("Remote UCB session finalized")